# Data extraction on a Colab GPU

Runs `cse-extract` on Colab so Docling uses a GPU, reading the PDFs from Google Drive and writing the results back to Drive.

1. **Runtime → Change runtime type → L4 GPU** (a T4 also works, more slowly).
2. Check the paths in the settings cell.
3. **Run all**, and approve the Google Drive prompt. Tick every permission box, or the mount fails.

Finished reports are skipped, so after a disconnect just run all again.

## Settings

In [ ]:
# Branch of ChunkWise-CSE to install the package from.
BRANCH = "data-extraction/dev"

# The Annual Reports folder as Colab sees it after mounting Drive. A folder synced by
# Drive for Desktop appears under /content/drive/Othercomputers/<computer name>/.
INPUT_DIR = "/content/drive/Othercomputers/My Laptop/Annual Reports"
OUTPUT_DIR = "/content/drive/MyDrive/chunkwise-cse/data-extraction/output"

LAYOUT_MODEL = None  # e.g. "layout_egret_large", or None for Docling's default

## Install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "cse-extract[docling] @ git+https://github.com/Hirukadeven07/ChunkWise-CSE@{BRANCH}#subdirectory=data-extraction"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: choose Runtime -> Change runtime type -> L4 GPU"
print(torch.cuda.get_device_name(0))

## Mount Drive and write the config

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import yaml

config_path = Path("/content/extraction.yaml")
config_path.write_text(yaml.safe_dump({
    "input_dir": INPUT_DIR,
    "output_dir": OUTPUT_DIR,
    "exclude_dirs": ["interim"],
    "docling": {"device": "cuda", "ocr": False, "table_mode": "accurate", "layout_model": LAYOUT_MODEL},
}))
!cse-extract list --config {config_path}

## Check the PDFs

Finds scanned (image-only) reports and duplicate files before using the GPU. Details go to `validation.csv` in the output folder.

In [ ]:
!cse-extract check --config {config_path}

## Extract

In [ ]:
!cse-extract run --config {config_path}

## Financial figures

Reads the figures from the extracted tables into `figures.csv` and `key_figures.csv` in the output folder.

In [ ]:
!cse-extract figures --config {config_path}

## Check the results

In [ ]:
import pandas as pd

manifest = pd.read_csv(Path(OUTPUT_DIR) / "manifest.csv")
print(manifest["status"].value_counts().to_string())
print(f"{manifest['pages'].sum()} pages, median {manifest['pages_per_second'].median():.2f} pages/s")
display(manifest[manifest["status"].isin(["failure", "error", "rejected"])][["report_id", "status", "error"]])

key = pd.read_csv(Path(OUTPUT_DIR) / "key_figures.csv")
key.head(20)